In [0]:
%python
# ====================================================================
# SILVER LAYER - STEP 1: CLEAN ALL BRONZE TABLES
# ====================================================================
# Purpose: Apply data quality checks and cleaning transformations
#          to all 9 bronze tables before joining them
# ====================================================================

from pyspark.sql.functions import (
    col, trim, lower, upper, to_timestamp, when, coalesce, 
    current_timestamp, regexp_replace, length, datediff
)
from pyspark.sql.types import DecimalType, IntegerType, DoubleType

# Project configuration
PROJECT_NAME = "retail"
CATALOG = "workspace"
BRONZE_SCHEMA = f"{PROJECT_NAME}_bronze"
SILVER_SCHEMA = f"{PROJECT_NAME}_silver"

print("=" * 70)
print("🧹 SILVER LAYER - CLEANING BRONZE TABLES")
print("=" * 70)
print(f"Source: {CATALOG}.{BRONZE_SCHEMA}")
print(f"Target: {CATALOG}.{SILVER_SCHEMA}")
print("=" * 70 + "\n")

In [0]:
%sql
-- Create silver schema if it doesn't exist
CREATE SCHEMA IF NOT EXISTS workspace.retail_silver
  COMMENT 'Cleaned and validated data layer';

-- Verify it exists
SHOW SCHEMAS IN workspace LIKE 'retail*';

In [0]:
%python
# ====================================================================
# HELPER FUNCTIONS - Reusable data cleaning logic
# ====================================================================

def remove_duplicates(df, subset_cols, description):
    """Remove duplicate rows based on specified columns."""
    initial_count = df.count()
    df_clean = df.dropDuplicates(subset=subset_cols)
    final_count = df_clean.count()
    duplicates_removed = initial_count - final_count
    
    print(f"  ├─ Duplicates removed: {duplicates_removed:,} rows")
    return df_clean


def handle_critical_nulls(df, critical_cols, description):
    """Drop rows where critical columns are null."""
    initial_count = df.count()
    
    # Build filter condition for all critical columns
    filter_condition = col(critical_cols[0]).isNotNull()
    for col_name in critical_cols[1:]:
        filter_condition = filter_condition & col(col_name).isNotNull()
    
    df_clean = df.filter(filter_condition)
    final_count = df_clean.count()
    rows_dropped = initial_count - final_count
    
    print(f"  ├─ Rows with critical nulls dropped: {rows_dropped:,}")
    return df_clean


def clean_string_column(df, col_name):
    """Trim whitespace and handle nulls in string columns."""
    return df.withColumn(
        col_name,
        when(col(col_name).isNotNull(), trim(col(col_name)))
        .otherwise(None)
    )


def parse_timestamp_column(df, col_name):
    """Convert string timestamps to proper timestamp type."""
    return df.withColumn(
        col_name,
        to_timestamp(col(col_name))
    )


def validate_positive_values(df, col_name):
    """Ensure numeric columns contain only positive values."""
    initial_count = df.count()
    df_clean = df.filter(col(col_name) > 0)
    final_count = df_clean.count()
    invalid_rows = initial_count - final_count
    
    if invalid_rows > 0:
        print(f"  ├─ Invalid {col_name} values removed: {invalid_rows:,} rows")
    
    return df_clean


def add_audit_columns(df):
    """Add metadata columns for tracking."""
    return df.withColumn("_cleaned_at", current_timestamp())


def write_to_silver(df, table_name):
    """Write cleaned DataFrame to silver schema."""
    full_table_name = f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"
    
    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(full_table_name))
    
    row_count = df.count()
    print(f"  └─ ✅ Written to {full_table_name} — {row_count:,} rows\n")

In [0]:
%python
# ====================================================================
# CLEAN: ORDERS
# ====================================================================
print("📦 Cleaning: bronze_orders")
print("-" * 70)

df_orders = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_orders")
print(f"  ├─ Initial rows: {df_orders.count():,}")

# Remove duplicates based on order_id
df_orders = remove_duplicates(df_orders, ["order_id"], "orders")

# Handle critical nulls - order_id and customer_id must exist
df_orders = handle_critical_nulls(df_orders, ["order_id", "customer_id"], "orders")

# Parse timestamp columns
timestamp_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for ts_col in timestamp_cols:
    df_orders = parse_timestamp_column(df_orders, ts_col)

# Clean order_status - trim and lowercase for consistency
df_orders = df_orders.withColumn(
    "order_status",
    lower(trim(col("order_status")))
)

# Calculate delivery metrics
df_orders = df_orders.withColumn(
    "days_to_delivery",
    datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp"))
)

df_orders = df_orders.withColumn(
    "is_late_delivery",
    when(
        col("order_delivered_customer_date") > col("order_estimated_delivery_date"),
        True
    ).otherwise(False)
)

# Add audit columns
df_orders = add_audit_columns(df_orders)

# Write to silver
write_to_silver(df_orders, "silver_orders")

In [0]:
%python
# ====================================================================
# CLEAN: CUSTOMERS
# ====================================================================
print("👥 Cleaning: bronze_customers")
print("-" * 70)

df_customers = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_customers")
print(f"  ├─ Initial rows: {df_customers.count():,}")

# Remove duplicates
df_customers = remove_duplicates(df_customers, ["customer_id"], "customers")

# Handle critical nulls
df_customers = handle_critical_nulls(df_customers, ["customer_id"], "customers")

# Clean string columns
string_cols = ["customer_unique_id", "customer_zip_code_prefix", "customer_city", "customer_state"]
for str_col in string_cols:
    df_customers = clean_string_column(df_customers, str_col)

# Standardize state codes to uppercase
df_customers = df_customers.withColumn(
    "customer_state",
    upper(col("customer_state"))
)

# Add audit columns
df_customers = add_audit_columns(df_customers)

# Write to silver
write_to_silver(df_customers, "silver_customers")

In [0]:
%python
# ====================================================================
# CLEAN: ORDER ITEMS
# ====================================================================
print("🛍️ Cleaning: bronze_order_items")
print("-" * 70)

df_order_items = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_order_items")
print(f"  ├─ Initial rows: {df_order_items.count():,}")

# Handle critical nulls
df_order_items = handle_critical_nulls(
    df_order_items, 
    ["order_id", "product_id", "seller_id"],
    "order_items"
)

# Validate positive prices and freight values
df_order_items = validate_positive_values(df_order_items, "price")
df_order_items = validate_positive_values(df_order_items, "freight_value")

# Calculate total item value
df_order_items = df_order_items.withColumn(
    "item_total_value",
    col("price") + col("freight_value")
)

# Parse shipping_limit_date
df_order_items = parse_timestamp_column(df_order_items, "shipping_limit_date")

# Add audit columns
df_order_items = add_audit_columns(df_order_items)

# Write to silver
write_to_silver(df_order_items, "silver_order_items")

In [0]:
%python
# ====================================================================
# CLEAN: PAYMENTS
# ====================================================================
print("💳 Cleaning: bronze_order_payments")
print("-" * 70)

df_payments = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_order_payments")
print(f"  ├─ Initial rows: {df_payments.count():,}")

# Handle critical nulls
df_payments = handle_critical_nulls(df_payments, ["order_id"], "payments")

# Clean payment_type - lowercase and trim
df_payments = df_payments.withColumn(
    "payment_type",
    lower(trim(col("payment_type")))
)

# Validate positive payment values
df_payments = validate_positive_values(df_payments, "payment_value")

# Handle installments - must be >= 1
df_payments = df_payments.filter(col("payment_installments") >= 1)

# Add audit columns
df_payments = add_audit_columns(df_payments)

# Write to silver
write_to_silver(df_payments, "silver_order_payments")

In [0]:
%python
# ====================================================================
# CLEAN: REVIEWS
# ====================================================================
print("⭐ Cleaning: bronze_order_reviews")
print("-" * 70)

df_reviews = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_order_reviews")
print(f"  ├─ Initial rows: {df_reviews.count():,}")

# Remove duplicates - keep first review per order
df_reviews = remove_duplicates(df_reviews, ["order_id"], "reviews")

# Handle critical nulls
df_reviews = handle_critical_nulls(df_reviews, ["review_id", "order_id"], "reviews")

# Validate review scores (must be 1-5)
initial_count = df_reviews.count()
df_reviews = df_reviews.filter(
    (col("review_score") >= 1) & (col("review_score") <= 5)
)
invalid_scores = initial_count - df_reviews.count()
if invalid_scores > 0:
    print(f"  ├─ Invalid review scores removed: {invalid_scores:,} rows")

# Parse timestamp columns
df_reviews = parse_timestamp_column(df_reviews, "review_creation_date")
df_reviews = parse_timestamp_column(df_reviews, "review_answer_timestamp")

# Handle null review comments - replace with placeholder
df_reviews = df_reviews.withColumn(
    "review_comment_message",
    coalesce(col("review_comment_message"), "No comment provided")
)

# Add sentiment flag based on score
df_reviews = df_reviews.withColumn(
    "review_sentiment",
    when(col("review_score") >= 4, "positive")
    .when(col("review_score") == 3, "neutral")
    .otherwise("negative")
)

# Add audit columns
df_reviews = add_audit_columns(df_reviews)

# Write to silver
write_to_silver(df_reviews, "silver_order_reviews")

In [0]:
# ====================================================================
# CLEAN: PRODUCTS
# ====================================================================
print("📦 Cleaning: bronze_products")
print("-" * 70)

df_products = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_products")
print(f"  ├─ Initial rows: {df_products.count():,}")

# Remove duplicates
df_products = remove_duplicates(df_products, ["product_id"], "products")

# Handle critical nulls
df_products = handle_critical_nulls(df_products, ["product_id"], "products")

# Clean string columns
df_products = clean_string_column(df_products, "product_category_name")

# Handle nulls in category - mark as 'unknown'
df_products = df_products.withColumn(
    "product_category_name",
    coalesce(col("product_category_name"), "unknown")
)

# Validate positive dimensions (filter out invalid products)
# Only keep products where measurements make sense
df_products = df_products.filter(
    (col("product_weight_g").isNull() | (col("product_weight_g") > 0)) &
    (col("product_length_cm").isNull() | (col("product_length_cm") > 0)) &
    (col("product_height_cm").isNull() | (col("product_height_cm") > 0)) &
    (col("product_width_cm").isNull() | (col("product_width_cm") > 0))
)

# Add audit columns
df_products = add_audit_columns(df_products)

# Write to silver
write_to_silver(df_products, "silver_products")

In [0]:
%python
# ====================================================================
# CLEAN: SELLERS
# ====================================================================
print("🏪 Cleaning: bronze_sellers")
print("-" * 70)

df_sellers = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_sellers")
print(f"  ├─ Initial rows: {df_sellers.count():,}")

# Remove duplicates
df_sellers = remove_duplicates(df_sellers, ["seller_id"], "sellers")

# Handle critical nulls
df_sellers = handle_critical_nulls(df_sellers, ["seller_id"], "sellers")

# Clean string columns
string_cols = ["seller_zip_code_prefix", "seller_city", "seller_state"]
for str_col in string_cols:
    df_sellers = clean_string_column(df_sellers, str_col)

# Standardize state codes
df_sellers = df_sellers.withColumn(
    "seller_state",
    upper(col("seller_state"))
)

# Add audit columns
df_sellers = add_audit_columns(df_sellers)

# Write to silver
write_to_silver(df_sellers, "silver_sellers")

In [0]:
%python
# ====================================================================
# CLEAN: GEOLOCATION
# ====================================================================
print("🗺️ Cleaning: bronze_geolocation")
print("-" * 70)

df_geo = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_geolocation")
print(f"  ├─ Initial rows: {df_geo.count():,}")

# Remove duplicates - keep one record per zip code
df_geo = remove_duplicates(
    df_geo, 
    ["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng"],
    "geolocation"
)

# Handle critical nulls
df_geo = handle_critical_nulls(
    df_geo,
    ["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng"],
    "geolocation"
)

# Validate latitude/longitude ranges
# Latitude: -90 to 90, Longitude: -180 to 180
initial_count = df_geo.count()
df_geo = df_geo.filter(
    (col("geolocation_lat") >= -90) & (col("geolocation_lat") <= 90) &
    (col("geolocation_lng") >= -180) & (col("geolocation_lng") <= 180)
)
invalid_coords = initial_count - df_geo.count()
if invalid_coords > 0:
    print(f"  ├─ Invalid coordinates removed: {invalid_coords:,} rows")

# Clean string columns
df_geo = clean_string_column(df_geo, "geolocation_city")
df_geo = clean_string_column(df_geo, "geolocation_state")

# Standardize state codes
df_geo = df_geo.withColumn(
    "geolocation_state",
    upper(col("geolocation_state"))
)

# Add audit columns
df_geo = add_audit_columns(df_geo)

# Write to silver
write_to_silver(df_geo, "silver_geolocation")

In [0]:
%python
# ====================================================================
# CLEAN: CATEGORY TRANSLATION
# ====================================================================
print("🔤 Cleaning: bronze_category_translation")
print("-" * 70)

df_cat_trans = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_category_translation")
print(f"  ├─ Initial rows: {df_cat_trans.count():,}")

# Remove duplicates
df_cat_trans = remove_duplicates(
    df_cat_trans,
    ["product_category_name"],
    "category_translation"
)

# Handle critical nulls
df_cat_trans = handle_critical_nulls(
    df_cat_trans,
    ["product_category_name", "product_category_name_english"],
    "category_translation"
)

# Clean string columns - trim and lowercase
df_cat_trans = clean_string_column(df_cat_trans, "product_category_name")
df_cat_trans = clean_string_column(df_cat_trans, "product_category_name_english")

# Replace spaces with underscores in English names for consistency
df_cat_trans = df_cat_trans.withColumn(
    "product_category_name_english",
    regexp_replace(col("product_category_name_english"), " ", "_")
)

# Add audit columns
df_cat_trans = add_audit_columns(df_cat_trans)

# Write to silver
write_to_silver(df_cat_trans, "silver_category_translation")

In [0]:
%python
# ====================================================================
# DATA QUALITY SUMMARY REPORT
# ====================================================================
print("\n" + "=" * 70)
print("📊 DATA QUALITY SUMMARY REPORT")
print("=" * 70 + "\n")

# Get row counts for all silver tables
silver_tables = [
    "silver_orders",
    "silver_customers",
    "silver_order_items",
    "silver_order_payments",
    "silver_order_reviews",
    "silver_products",
    "silver_sellers",
    "silver_geolocation",
    "silver_category_translation"
]

print(f"{'Table Name':<40} {'Row Count':>15}")
print("-" * 70)

total_rows = 0
for table_name in silver_tables:
    full_name = f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"
    count = spark.table(full_name).count()
    total_rows += count
    print(f"{table_name:<40} {count:>15,}")

print("-" * 70)
print(f"{'TOTAL ROWS IN SILVER LAYER':<40} {total_rows:>15,}")
print("=" * 70)

print("\n✅ Silver Layer - Step 1 Complete: All bronze tables cleaned and validated!")
print("\nNext Step: Run 02_join_enriched_orders.py to create the master fact table\n")

In [0]:
%sql
-- Verify all silver tables exist
SHOW TABLES IN workspace.retail_silver;

In [0]:
%sql
-- Quick peek at cleaned orders table
SELECT 
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp,
    days_to_delivery,
    is_late_delivery,
    _cleaned_at
FROM workspace.retail_silver.silver_orders
LIMIT 10;